# Product Image URL Analysis

Comprehensive analysis of product data from `data/site-content.json`:
- Extract all products with their IDs, names, image URLs, and images array
- Identify duplicate image URLs across products
- Detect suspicious image sharing patterns among recently added products

## 1. Import Required Libraries

In [1]:
import json
import pandas as pd
from collections import defaultdict
from pathlib import Path

# Display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 2. Load JSON Data from File

In [2]:
# Load the JSON data
json_path = Path('data/site-content.json')
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"✓ Loaded data from {json_path}")
print(f"✓ Total products: {len(data['products'])}")
print(f"✓ Data structure keys: {list(data.keys())}")


✓ Loaded data from data\site-content.json
✓ Total products: 40
✓ Data structure keys: ['storeName', 'storeDescription', 'footerText', 'logoUrl', 'features', 'home', 'promoSlides', 'products', 'services', 'flashSale', 'updatedAt']


## 3. Extract Product Information

In [3]:
# Extract all product data with their image information
products_data = []

for product in data['products']:
    product_info = {
        'id': product.get('id', 'N/A'),
        'name': product.get('name', 'N/A'),
        'primary_image_url': product.get('image', 'N/A'),
        'images_array_count': len(product.get('images', [])),
        'images_array': product.get('images', []),
        'category': product.get('category', 'N/A')
    }
    products_data.append(product_info)

# Create DataFrame
df_products = pd.DataFrame(products_data)

print(f"Total products extracted: {len(df_products)}")
print("\n" + "="*100)
print("PRODUCT SUMMARY TABLE")
print("="*100)

# Display key columns
summary_df = df_products[['id', 'name', 'category', 'images_array_count', 'primary_image_url']].copy()
print(summary_df.to_string())


Total products extracted: 40

PRODUCT SUMMARY TABLE
                                                                                                            id                                                                                                                                                                                 name          category  images_array_count                                                                                                                                                                            primary_image_url
0                                                                                               product-5oxd0b                                                                                                            Vintage Table Clock for Home, Silent Melting Clock, Kitchen, Office Table        Appliances                   1                                                                             https://res.cloudinary.

## 4. Identify Duplicate Image URLs

In [4]:
# Create a mapping of all image URLs (primary + array images) to products
image_url_map = defaultdict(list)

# Map primary images
for idx, row in df_products.iterrows():
    primary_url = row['primary_image_url']
    if primary_url and primary_url != 'N/A':
        image_url_map[primary_url].append({
            'product_id': row['id'],
            'product_name': row['name'],
            'image_type': 'PRIMARY',
            'category': row['category']
        })

# Map images from array
for idx, row in df_products.iterrows():
    images_array = row['images_array']
    if images_array:
        for img_url in images_array:
            if img_url and img_url != 'N/A':
                image_url_map[img_url].append({
                    'product_id': row['id'],
                    'product_name': row['name'],
                    'image_type': 'ARRAY',
                    'category': row['category']
                })

# Find duplicate image URLs (URLs used by multiple products)
duplicate_urls = {url: products for url, products in image_url_map.items() if len(products) > 1}

print(f"\n{'='*100}")
print(f"IMAGE URL DUPLICATION ANALYSIS")
print(f"{'='*100}")
print(f"\nTotal unique image URLs: {len(image_url_map)}")
print(f"Duplicate image URLs (used by multiple products): {len(duplicate_urls)}")

if len(duplicate_urls) > 0:
    print("\n⚠️  DUPLICATE IMAGE URLs DETECTED:")
    print("-" * 100)
    for idx, (url, products) in enumerate(duplicate_urls.items(), 1):
        print(f"\n{idx}. URL: {url}")
        print(f"   Used by {len(products)} products:")
        for prod in products:
            print(f"      • {prod['product_id']} | {prod['product_name'][:60]} | Type: {prod['image_type']}")
else:
    print("\n✓ No duplicate image URLs found across products.")



IMAGE URL DUPLICATION ANALYSIS

Total unique image URLs: 81
Duplicate image URLs (used by multiple products): 0

✓ No duplicate image URLs found across products.


## 5. Analyze Image Sharing Patterns for Newly Added Products

In [5]:
# Identify recently added products (minimal images array - typically 1 image)
df_products['is_newly_added'] = df_products['images_array_count'] == 1

newly_added = df_products[df_products['is_newly_added']]
established = df_products[~df_products['is_newly_added']]

print(f"\n{'='*100}")
print(f"NEWLY ADDED PRODUCTS ANALYSIS")
print(f"{'='*100}")
print(f"\nProducts with minimal images (1 image in array): {len(newly_added)}")
print(f"Products with multiple images (2+ images in array): {len(established)}")

# Check if newly added products share image URLs among themselves
newly_added_image_sharing = {}
for url, products in duplicate_urls.items():
    newly_added_using_url = [p for p in products if any(
        p['product_id'] == pid for pid in newly_added['id']
    )]
    if len(newly_added_using_url) > 1:
        newly_added_image_sharing[url] = newly_added_using_url

if newly_added_image_sharing:
    print(f"\n⚠️  IMAGE SHARING AMONG NEWLY ADDED PRODUCTS:")
    print("-" * 100)
    for url, products in newly_added_image_sharing.items():
        print(f"\nURL: {url}")
        print(f"Shared by {len(products)} newly added products:")
        for prod in products:
            print(f"   • {prod['product_id']} | {prod['product_name'][:50]}")
else:
    print(f"\n✓ No image sharing detected among newly added products.")

# Identify products with more images for comparison
print(f"\n{'='*100}")
print(f"PRODUCTS WITH MULTIPLE IMAGES (Most Detailed)")
print(f"{'='*100}")
if len(established) > 0:
    multi_image_df = established.sort_values('images_array_count', ascending=False)
    for idx, row in multi_image_df.iterrows():
        print(f"\n• {row['id']} | {row['name'][:60]}")
        print(f"  Images in array: {row['images_array_count']}")
        print(f"  Primary: {row['primary_image_url']}")
        for i, img in enumerate(row['images_array'], 1):
            print(f"  Array[{i}]: {img}")
else:
    print("\nNo products with multiple images found.")



NEWLY ADDED PRODUCTS ANALYSIS

Products with minimal images (1 image in array): 39
Products with multiple images (2+ images in array): 1

✓ No image sharing detected among newly added products.

PRODUCTS WITH MULTIPLE IMAGES (Most Detailed)

• product-2ok3yz | EAGEAT Unisex Anti Blue Light Protective Computer Screen Gla
  Images in array: 2
  Primary: https://res.cloudinary.com/dz5jwwdhw/image/upload/v1775643095/gadget-hub/products/product-2ok3yz/file_tvquhw.jpg
  Array[1]: https://res.cloudinary.com/dz5jwwdhw/image/upload/v1775643103/gadget-hub/products/product-2ok3yz/file_t7c1qr.jpg
  Array[2]: https://res.cloudinary.com/dz5jwwdhw/image/upload/v1775643601/gadget-hub/products/product-2ok3yz/file_kh7dff.jpg


## 6. Generate Summary Report

In [6]:
print(f"\n{'='*100}")
print(f"COMPREHENSIVE SUMMARY REPORT")
print(f"{'='*100}\n")

# 1. Overall Statistics
print("📊 OVERALL STATISTICS:")
print("-" * 100)
print(f"Total Products: {len(df_products)}")
print(f"Total Unique Image URLs: {len(image_url_map)}")
print(f"Duplicate Image URLs: {len(duplicate_urls)}")
print(f"Newly Added Products (1 image): {len(newly_added)}")
print(f"Established Products (2+ images): {len(established)}")

# 2. Image Distribution
print(f"\n📈 IMAGE DISTRIBUTION:")
print("-" * 100)
print(f"Products with 1 image: {len(df_products[df_products['images_array_count'] == 1])}")
print(f"Products with 2 images: {len(df_products[df_products['images_array_count'] == 2])}")
print(f"Products with 3+ images: {len(df_products[df_products['images_array_count'] >= 3])}")
print(f"Average images per product: {df_products['images_array_count'].mean():.2f}")

# 3. Complete Product Listing with Primary Image URLs
print(f"\n📚 COMPLETE PRODUCT LISTING WITH PRIMARY IMAGE URLS:")
print("-" * 100)
for idx, row in df_products.iterrows():
    print(f"\n{idx+1}. {row['id']}")
    print(f"   Name: {row['name']}")
    print(f"   Category: {row['category']}")
    print(f"   Images in array: {row['images_array_count']}")
    print(f"   Primary Image URL: {row['primary_image_url']}")

# 4. Duplicate Alert Summary
print(f"\n\n{'='*100}")
print(f"⚠️  DUPLICATE & SUSPICIOUS PATTERNS SUMMARY")
print(f"{'='*100}")

if len(duplicate_urls) == 0:
    print("\n✅ CLEAN: No duplicate image URLs detected across products.")
    print("   All images are unique to their respective products.")
else:
    print(f"\n❌ WARNING: {len(duplicate_urls)} duplicate image URL(s) found!")
    print("\nDetails:")
    for url, products in duplicate_urls.items():
        print(f"\n  • {url}")
        print(f"    Products using this URL: {len(products)}")
        for p in products:
            print(f"      - {p['product_id']}: {p['product_name'][:50]}")

# 5. Recommendations
print(f"\n\n{'='*100}")
print(f"💡 RECOMMENDATIONS")
print(f"{'='*100}")
if len(duplicate_urls) > 0:
    print("\n1. Review products with duplicate images - they may be:")
    print("   - Unintentional duplicates")
    print("   - Variants of the same product")
    print("   - Incorrectly linked images")
    print("\n2. Consider consolidating or relinking images to appropriate products")
    print("3. Verify image URLs are correctly pointing to intended products")
else:
    print("\n✓ Image URL strategy is clean - no duplicates detected")
    print("✓ All images are properly isolated to their respective products")
    print("✓ Continue monitoring for new duplicate patterns as products are added")

print(f"\n{'='*100}")
print("Report generated successfully")
print(f"{'='*100}\n")



COMPREHENSIVE SUMMARY REPORT

📊 OVERALL STATISTICS:
----------------------------------------------------------------------------------------------------
Total Products: 40
Total Unique Image URLs: 81
Duplicate Image URLs: 0
Newly Added Products (1 image): 39
Established Products (2+ images): 1

📈 IMAGE DISTRIBUTION:
----------------------------------------------------------------------------------------------------
Products with 1 image: 39
Products with 2 images: 1
Products with 3+ images: 0
Average images per product: 1.02

📚 COMPLETE PRODUCT LISTING WITH PRIMARY IMAGE URLS:
----------------------------------------------------------------------------------------------------

1. product-5oxd0b
   Name: Vintage Table Clock for Home, Silent Melting Clock, Kitchen, Office Table
   Category: Appliances
   Images in array: 1
   Primary Image URL: https://res.cloudinary.com/dz5jwwdhw/image/upload/v1775649642/gadget-hub/products/product-5oxd0b/file_m2a8xt.jpg

2. product-0g4pit
   Name: 8MP